# Step 4 — Hyperparameter Tuning (ResNet50)
**Oral Diseases Image Classification · Sprint Two**

ResNet50 was the best model in Step 3 (val macro-F1 0.870 vs 0.842 / 0.843), so we tune it.

**What we search** (a small random search, 8 trials max):
| Hyperparameter | Values |
|---|---|
| Fine-tuning learning rate | 1e-5, 3e-5, 1e-4 |
| Unfrozen layers (from the end) | 30, 80, 140 |
| Dropout | 0.3, 0.5 |
| Batch size | 16, 32 |

- **Trial 0 is the Step 3 setup** (1e-5 / 30 / 0.3 / 32). It is our reference, and it also shows how much a re-run changes *by chance alone* (GPU training is not perfectly repeatable).
- Everything is judged on the **validation** set; the **test set is still untouched**.
- ⚠️ The validation set has only 498 images. With 8 trials we are partly "fitting" the validation set, so the best trial's score will be a bit optimistic, and **differences below ~1.5 points of macro-F1 are within noise**. The honest number comes from the test set at the end.
- A time guard stops launching new trials after 90 minutes.

**Before running**
1. *Settings → Accelerator → **GPU T4 x2***, *Internet → **ON***
2. *Add Input → Notebooks → Your Work →* the **Step 1** notebook
3. Best is **Save Version → Save & Run All (Commit)** directly and let it run in the background (≈ 60–90 min).

In [ ]:
import os, gc, json, time, random, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, accuracy_score

SEED, IMG_SIZE, BATCH_EVAL = 42, (224, 224), 32
EPOCHS_HEAD = int(os.environ.get("EPOCHS_HEAD", 8))
EPOCHS_FT   = int(os.environ.get("EPOCHS_FT", 25))
N_TRIALS    = int(os.environ.get("N_TRIALS", 8))
TIME_BUDGET_MIN = float(os.environ.get("TIME_BUDGET_MIN", 90))
LR_HEAD     = 1e-3
WEIGHTS     = None if os.environ.get("WEIGHTS") == "none" else "imagenet"

SEARCH_SPACE = {"lr_ft": [1e-5, 3e-5, 1e-4], "unfreeze_last": [30, 80, 140], "dropout": [0.3, 0.5], "batch": [16, 32]}
BASELINE     = {"lr_ft": 1e-5, "unfreeze_last": 30, "dropout": 0.3, "batch": 32}      # = Step 3 setup

SEARCH_ROOT = os.environ.get("DATA_ROOT", "/kaggle/input")
WORK_DIR    = os.environ.get("WORK_DIR", "/kaggle/working")
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

DATA_DIR = None
for d, _, files in os.walk(SEARCH_ROOT):
    if "manifest.csv" in files and "meta.json" in files:
        DATA_DIR = d; break
assert DATA_DIR, "Step-1 output not found. Add it via: Add Input -> Notebooks -> Your Work"
CLASS_NAMES = json.load(open(os.path.join(DATA_DIR, "meta.json")))["class_names"]
for sub in ("results", "models"): os.makedirs(os.path.join(WORK_DIR, sub), exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPUs      :", tf.config.list_physical_devices("GPU") or "NONE  <-- turn on the GPU accelerator!")
print("DATA_DIR  :", DATA_DIR)

# trial list: baseline first, then random distinct configs from the grid
grid   = [dict(zip(SEARCH_SPACE, v)) for v in itertools.product(*SEARCH_SPACE.values())]
others = [g for g in grid if g != BASELINE]
random.Random(SEED).shuffle(others)
TRIALS = [BASELINE] + others[:N_TRIALS - 1]
print(f"\nPlanned trials ({len(TRIALS)} of {len(grid)} possible configs):")
print(pd.DataFrame(TRIALS).rename_axis("trial").to_string())

## 1) Data and model builders (same as Step 3)

In [ ]:
def make_ds(split, shuffle, batch):
    ds = keras.utils.image_dataset_from_directory(
        os.path.join(DATA_DIR, split), label_mode="categorical", class_names=CLASS_NAMES,
        image_size=IMG_SIZE, batch_size=batch, shuffle=shuffle, seed=SEED)
    return ds.prefetch(tf.data.AUTOTUNE)

val_ds = make_ds("val", False, BATCH_EVAL)
_train_cache = {}
def get_train_ds(batch):
    if batch not in _train_cache: _train_cache[batch] = make_ds("train", True, batch)
    return _train_cache[batch]

counts = np.array([len(os.listdir(os.path.join(DATA_DIR, "train", c))) for c in CLASS_NAMES])
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(counts.sum() / (len(counts) * counts))}

@keras.utils.register_keras_serializable(package="oral")
class ImageNetPreprocess(layers.Layer):
    def __init__(self, mode="none", **kwargs):
        super().__init__(**kwargs); self.mode = mode
    def call(self, x):
        if self.mode == "none":
            return x
        return keras.applications.imagenet_utils.preprocess_input(x, mode=self.mode)
    def get_config(self):
        cfg = super().get_config(); cfg.update(mode=self.mode); return cfg

def build_resnet(dropout, weights=WEIGHTS):
    try:
        base = ResNet50(include_top=False, weights=weights, input_shape=IMG_SIZE + (3,))
    except Exception as e:
        raise RuntimeError("Could not load ImageNet weights. Turn Settings -> Internet ON and re-run.") from e
    base.trainable = False
    inp = keras.Input(IMG_SIZE + (3,))
    x = layers.RandomFlip("horizontal")(inp)
    x = layers.RandomRotation(0.08)(x)
    x = layers.RandomZoom(0.1)(x)
    x = layers.RandomContrast(0.1)(x)
    x = ImageNetPreprocess("caffe")(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(len(CLASS_NAMES), activation="softmax")(x)
    return keras.Model(inp, out, name="ResNet50_clf"), base

def unfreeze_last(base, n):
    base.trainable = True
    for l in base.layers[:-n]:
        l.trainable = False
    for l in base.layers:
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False

def save_without_optimizer(model, dropout, path):
    # copy the weights into a fresh, never-compiled model and save that: ~1/3 the size, enough for the app
    clean, _ = build_resnet(dropout, weights=None)
    clean.set_weights(model.get_weights())
    clean.save(path)
    return clean

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="categorical_crossentropy", metrics=["accuracy"])

## 2) One trial = head training, then fine-tuning with the trial's settings

In [ ]:
def run_trial(i, cfg):
    keras.backend.clear_session(); gc.collect(); tf.random.set_seed(SEED)
    print(f"\n{'='*78}\nTRIAL {i}: {cfg}\n{'='*78}")
    t0 = time.time()
    model, base = build_resnet(cfg["dropout"])
    train_ds = get_train_ds(cfg["batch"])

    compile_model(model, LR_HEAD)
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, class_weight=CLASS_WEIGHT, verbose=2, callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)])

    unfreeze_last(base, cfg["unfreeze_last"])
    compile_model(model, cfg["lr_ft"])
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, class_weight=CLASS_WEIGHT, verbose=2, callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7)])

    y_true = np.concatenate([y.numpy().argmax(1) for _, y in val_ds])
    y_pred = model.predict(val_ds, verbose=0).argmax(1)
    row = {"trial": i, **cfg, "val_accuracy": accuracy_score(y_true, y_pred),
           "val_macro_f1": f1_score(y_true, y_pred, average="macro"),
           "best_val_loss": float(min(h2.history["val_loss"])),
           "epochs_ft": len(h2.history["loss"]), "minutes": round((time.time() - t0) / 60, 1)}
    print(f"\n>>> TRIAL {i} done: val acc {row['val_accuracy']:.3f} | macro-F1 {row['val_macro_f1']:.3f} | {row['minutes']} min")
    return row, model, y_true, y_pred

## 3) Run the search

In [ ]:
rows, best = [], {"f1": -1}
T0 = time.time()
for i, cfg in enumerate(TRIALS):
    if i > 0 and (time.time() - T0) / 60 > TIME_BUDGET_MIN:
        print(f"\nTime budget of {TIME_BUDGET_MIN:.0f} min reached - skipping trials {i}..{len(TRIALS)-1}")
        break
    row, model, y_true, y_pred = run_trial(i, cfg)
    rows.append(row)
    pd.DataFrame(rows).to_csv(os.path.join(WORK_DIR, "results", "tuning_resnet50.csv"), index=False)
    if row["val_macro_f1"] > best["f1"]:
        path = os.path.join(WORK_DIR, "models", "ResNet50_tuned_best.keras")
        save_without_optimizer(model, cfg["dropout"], path)
        best = {"f1": row["val_macro_f1"], "trial": i, "cfg": cfg, "path": path, "y_true": y_true, "y_pred": y_pred,
                "params": int(model.count_params())}
        print(f"    new best -> saved to {path}")
    del model; gc.collect()
print(f"\nTotal search time: {(time.time()-T0)/60:.1f} min")

## 4) Results of the search

In [ ]:
res = pd.DataFrame(rows)
show = res.sort_values("val_macro_f1", ascending=False).copy()
for c in ("val_accuracy", "val_macro_f1", "best_val_loss"): show[c] = show[c].round(3)
print(show.to_string(index=False))

base_f1 = res.loc[res.trial == 0, "val_macro_f1"].iloc[0]
print(f"\nTrial 0 (Step 3 setup, re-run): macro-F1 {base_f1:.3f}   (Step 3 run gave 0.870 -> re-run noise = {abs(base_f1-0.870):.3f})")
print(f"Best trial: {best['trial']}  macro-F1 {best['f1']:.3f}  config {best['cfg']}")
print(f"Gain over trial 0: {best['f1']-base_f1:+.3f}   (differences under ~0.015 are within noise for a 498-image validation set)")

colors = ["#999999" if t == 0 else ("#2E7D32" if t == best["trial"] else "#4C78A8") for t in res.trial]
plt.figure(figsize=(8, 3.6)); plt.bar(res.trial.astype(str), res.val_macro_f1, color=colors)
plt.ylim(max(0, res.val_macro_f1.min() - 0.05), min(1, res.val_macro_f1.max() + 0.03))
plt.xlabel("trial (grey = Step 3 setup, green = best)"); plt.ylabel("val macro-F1"); plt.title("ResNet50 tuning")
plt.tight_layout(); plt.show()

## 5) The best model in detail (validation set)

In [ ]:
y_true, y_pred = best["y_true"], best["y_pred"]
labels = list(range(len(CLASS_NAMES)))
cm = confusion_matrix(y_true, y_pred, labels=labels)
print(classification_report(y_true, y_pred, labels=labels, target_names=CLASS_NAMES, digits=3, zero_division=0))

print("Most frequent mistakes (true class -> predicted class: count):")
pairs = sorted([(CLASS_NAMES[i], CLASS_NAMES[j], int(cm[i, j])) for i in labels for j in labels if i != j and cm[i, j] > 0], key=lambda t: -t[2])
for t, p, n in pairs[:8]: print(f"  {t:14s} -> {p:14s}: {n}")

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax[0], xticks_rotation=45, colorbar=False); ax[0].set_title("Tuned ResNet50 - confusion matrix (val)")
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=labels, normalize="true").round(2), display_labels=CLASS_NAMES).plot(ax=ax[1], xticks_rotation=45, colorbar=False)
ax[1].set_title("row-normalised (= recall)")
plt.tight_layout(); plt.show()

In [ ]:
best_model = keras.models.load_model(best["path"])
t1 = time.time(); best_model.predict(val_ds, verbose=0); ms_img = (time.time() - t1) / len(y_true) * 1000
recall = {c: float(r) for c, r in zip(CLASS_NAMES, cm.diagonal() / np.maximum(cm.sum(1), 1))}
result = {"model": "ResNet50_tuned", "split": "val", "val_accuracy": float(accuracy_score(y_true, y_pred)),
          "val_macro_f1": float(f1_score(y_true, y_pred, average="macro")), "per_class_recall": recall,
          "config": best["cfg"], "params": best["params"], "trials_run": len(rows),
          "model_size_mb": round(os.path.getsize(best["path"]) / 1e6, 1), "ms_per_image": round(ms_img, 2)}
json.dump(result, open(os.path.join(WORK_DIR, "results", "ResNet50_tuned_val.json"), "w"), indent=2)
print(json.dumps(result, indent=2))